# Jalon 1 : comprendre et exécuter les parties 1 à 3

Ce notebook accompagne les scripts du projet. Il lance les mêmes traitements, puis affiche leurs preuves. Les résultats sont produits par Spark, et ne sont pas inscrits à l'avance dans le notebook.

Le noyau Jupyter joue le rôle de Driver ; le Master standalone affecte les ressources des deux Workers. Les données sont lues depuis MinIO, puis Silver est écrit en Parquet partitionné par ville.


In [ ]:
from pathlib import Path
import sys, json
from argparse import Namespace

PROJECT = Path('/workspace')
if not PROJECT.exists():
    PROJECT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT))
from src.analysis import run_analysis


## Avant de lancer

Le générateur doit avoir produit 50 000 événements et les commandes de chargement du README doivent avoir déposé l'historique dans Bronze ainsi que les 200 restaurants dans le référentiel. Le README donne les commandes exactes.

Le traitement conserve les doublons pour les KPI pédagogiques RDD, DataFrame et SQL afin de comparer les mêmes données. Il les supprime ensuite pour l'export Silver. Un `DELIVERED` représente un événement de livraison, et son doublon n'est pas une seconde commande réelle.


In [ ]:
args = Namespace(
    orders='s3a://lake/bronze/history/orders_history.jsonl',
    restaurants='s3a://lake/reference/restaurants.json',
    silver='s3a://lake/silver/orders_history',
    evidence_dir=str(PROJECT / 'evidence'),
    master='spark://spark-master:7077',
    app_name='Jalon1-Notebook',
    repetitions=3,
    hold_seconds=0,
)
results = run_analysis(args)  # Ferme sa SparkSession à la fin ; journaux conservés.


## Partie 2 : transformations, actions et précision

Le code suit `sc.textFile → map(parse_order) → filter(DELIVERED) → map → reduceByKey`. Le CA est additionné en centimes entiers pour éviter les erreurs d'arrondi binaire. Le panier moyen utilise une paire `(somme, nombre)` dans un seul `reduceByKey`.

`textFile`, `map`, `filter` et `reduceByKey` construisent le graphe sans exécuter immédiatement le calcul : ce sont des transformations. `count`, `collect` et `takeOrdered` déclenchent les jobs : ce sont des actions. `toDebugString` décrit le lineage sans déclencher un calcul.


In [ ]:
from IPython.display import HTML, display
from html import escape

def display_table(rows):
    if not rows:
        print('(aucune ligne)')
        return
    fields = list(rows[0])
    head = ''.join('<th>' + escape(str(k)) + '</th>' for k in fields)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(row.get(k, ''))) + '</td>' for k in fields) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))


display_table(results['raw_city_kpis'])
display_table(results['raw_top5_restaurants'])
print((PROJECT / 'evidence/rdd_lineage.txt').read_text(encoding='utf-8'))


`reduceByKey` regroupe les valeurs d'une même ville. Un combine local réduit les échanges, puis le shuffle réunit les clés entre partitions : cette dépendance sépare les stages. Les identifiants réels sont enregistrés ci-dessous ; consulter le History Server pour capturer le DAG d'un job précis.

La tolérance aux pannes vient du lineage : si une partition est perdue, Spark peut la reconstruire à partir des entrées et des transformations. Cela suppose que la source reste disponible et que les transformations puissent être rejouées.


In [ ]:
actions = results['actions']
fields = ['label', 'job_ids', 'stage_ids', 'stage_count', 'seconds']
display_table([{key: row[key] for key in fields} for row in actions if row['label'] in ['rdd_revenue_by_city', 'rdd_sum_count_one_pass', 'rdd_top5']])


## Partie 3 : schémas, SQL et jointure

Les deux fichiers JSON sont lus avec un `StructType` explicite : aucune inférence n'est utilisée. `DecimalType(12,2)` conserve la précision financière et `TimestampType` représente l'heure de l'événement en UTC.

La même somme est calculée en DataFrame puis dans `spark.sql` après `createOrReplaceTempView`. Le programme vérifie leur égalité exacte avec le résultat RDD. La jointure gauche conserve les commandes qui n'ont pas de restaurant correspondant et les comptabilise comme anomalie de qualité. Le petit référentiel est diffusé avec `broadcast` pour éviter de redistribuer tout l'historique.


In [ ]:
print('Réconciliation RDD / DataFrame / SQL :', results['reconciliation_rdd_dataframe_sql'])
display_table(results['raw_revenue_category_city'])
print((PROJECT / 'evidence/explain_join_extended.txt').read_text(encoding='utf-8'))


Le fichier `explain_join_extended.txt` contient le plan logique parsé, le plan logique analysé (types et colonnes résolus), le plan optimisé par Catalyst et le plan physique. La stratégie attendue ici est `BroadcastHashJoin`, car le code diffuse explicitement les 200 restaurants. Le plan réellement produit fait foi.

### UDF et expression native

La bande vaut `court` si la distance est strictement inférieure à 2 km, `moyen` de 2 à 5 km inclus et `long` au-delà de 5 km. Les distances nulles ou négatives ne reçoivent pas de bande. La variante `when/otherwise` évite le passage entre la JVM et Python et expose les conditions au moteur SQL.

Le benchmark groupe réellement par la bande calculée et collecte les effectifs. Un simple `count()` des lignes serait insuffisant : Catalyst pourrait éliminer une colonne calculée inutilisée. Chaque variante a un tour de chauffe, puis trois mesures alternées ; leurs médianes restent indicatives sur ce petit jeu.


In [ ]:
display_table([{'api': key, **value} for key, value in results['benchmark']['api'].items()])
display_table([{'variant': key, **value} for key, value in results['benchmark']['distance'].items()])
print('Bandes :', results['distance_bands'])


### Silver et limites d'interprétation

Silver supprime les doublons par `(order_id,status)` puis conserve tous les statuts, les colonnes du référentiel et la bande de distance. Une écriture Parquet partitionnée par `city` permet l'élagage des partitions pour les lectures filtrées sur une ville. Le programme relit ensuite Silver et vérifie son nombre de lignes.

Le typage objet des `Dataset[T]` est disponible en Scala et Java. En PySpark, le DataFrame est une table structurée contrôlée par le schéma, sans API `Dataset[T]` équivalente. Les RDD exposent davantage le traitement manuel ; les DataFrames et SQL bénéficient de Catalyst et du moteur d'exécution optimisé de Spark. Les temps mesurés incluent lecture, traitement et transfert des petits agrégats au Driver, et ne constituent pas une comparaison de production.

Le sujet du jalon 1 ne fournit pas `tenant_id` dans les commandes : aucune preuve d'isolation multi-tenant ne peut être déduite de ce jeu.


In [ ]:
display_table(results['clean_city_kpis'])
print(json.dumps(results['quality'], ensure_ascii=False, indent=2))
print((PROJECT / 'evidence/silver_schema.txt').read_text(encoding='utf-8'))
